# Customer Retention Intelligence — MLflow Tracking & Model Registry (Phase 10)

## Objective

Make the already-selected Phase 7 model reproducible and trackable:

```
data/features -> model config -> training (already done, Phase 7)
    -> metrics -> model artifact -> MLflow tracking
    -> MLflow Model Registry -> versioned, reloadable model
```

**This notebook does not retrain, retune, or change the Phase 7 model in
any way.** The selected model remains exactly:

```
LogisticRegression(C=0.01, class_weight="balanced")
```

Every step below loads the frozen `models/tuned_logistic_regression.joblib`
(via `src/models/explainability.load_tuned_pipeline`, which already
refuses to load anything with different hyperparameters) and only
tracks/registers/reloads it. All logic lives in
`src/models/mlflow_tracking.py`; this notebook calls it and inspects the
results.

**Tracking backend:** a local SQLite database
(`configs/mlflow_config.yaml`), because the Model Registry requires a
database-backed store and SQLite needs no server or cloud infrastructure.
Both `mlruns.db` and `mlartifacts/` are generated locally, excluded from
version control, and fully reproducible from this config plus the
already-tracked Phase 7 artifact and reports -- they are not committed.

**On the "final holdout" test metrics logged below:** they are logged
*only* for the permanent record, tagged `test_metrics_are_final_holdout_only`.
Phase 7 already selected this model using **validation PR-AUC alone**,
before these numbers were ever computed here. Nothing in this notebook
uses them to choose or tune anything.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

import mlflow
from src.models import baseline as bl
from src.models.explainability import load_tuned_pipeline
from src.models import mlflow_tracking as mt

print(f"MLflow version: {mlflow.__version__}")
PROJECT_ROOT

2026/09/23 11:37:30 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/dhananjaymandal/customer-retention-intelligence/.venv/lib/python3.14/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


MLflow version: 3.16.1


PosixPath('/Users/dhananjaymandal/customer-retention-intelligence')

## 1. MLflow Setup

Load the Phase 10 config and point MLflow at the local SQLite-backed
tracking store.

In [2]:
config = mt.load_config()
tracking_uri = mt.configure_tracking(config)

print(f"Tracking URI: {tracking_uri}")
print(f"Experiment name: {config['experiment']['name']}")
print(f"Registry model name: {config['registry']['model_name']}")
print(f"Expected model config: {config['expected_model']}")

Tracking URI: sqlite:////Users/dhananjaymandal/customer-retention-intelligence/mlruns.db
Experiment name: customer-churn-prediction
Registry model name: customer-churn-logistic-regression
Expected model config: {'model_type': 'LogisticRegression', 'C': 0.01, 'class_weight': 'balanced'}


## 2. Experiment Creation / Loading

Idempotent: re-running this cell later reuses the same experiment instead
of creating a duplicate.

In [3]:
experiment_id = mt.create_mlflow_experiment(config=config)
experiment = mlflow.get_experiment(experiment_id)

print(f"Experiment ID: {experiment.experiment_id}")
print(f"Experiment name: {experiment.name}")
print(f"Artifact location: {experiment.artifact_location}")

2026/09/23 11:37:33 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/09/23 11:37:33 INFO mlflow.store.db.utils: Updating database tables


Experiment ID: 1
Experiment name: customer-churn-prediction
Artifact location: /Users/dhananjaymandal/customer-retention-intelligence/mlartifacts


## 3. Load the Frozen Phase 7 Model and Data

`load_tuned_pipeline` (Phase 8, reused unmodified) refuses to load
anything whose hyperparameters aren't exactly `C=0.01`,
`class_weight="balanced"`.

In [4]:
pipeline = load_tuned_pipeline()
model = pipeline.named_steps["model"]
print(pipeline)
print(f"\nC={model.C}, class_weight={model.class_weight}, solver={model.solver}, random_state={model.random_state}")

features = bl.load_feature_dataset()
splits = bl.split_by_snapshot_date(features)
numeric_columns, categorical_columns = bl.get_feature_columns(features)
feature_columns = numeric_columns + categorical_columns

X_train, y_train = splits["train"][feature_columns], splits["train"][bl.TARGET_COLUMN]
X_val, y_val = splits["validation"][feature_columns], splits["validation"][bl.TARGET_COLUMN]
X_test, y_test = splits["test"][feature_columns], splits["test"][bl.TARGET_COLUMN]

print(f"\ntrain={len(X_train)}, validation={len(X_val)}, test={len(X_test)}, raw features={len(feature_columns)}")

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'tenure_days',
                                                   'tenure_months',
                                                   'txn_count_30d',
                                                   'txn_amount_sum_30d',
                                                   'txn_amount_mean_30d',
                                                   'txn_amount_max_30d',
                                                   'txn_success_count_30d',
                                                   'txn_


train=85419, validation=26337, test=24945, raw features=131


## 4. Build Parameters, Metrics, and Reproducibility Tags

Parameters cover model type, hyperparameters, solver, random seed,
feature count, and the training/validation date ranges. Metrics cover
training and validation performance (PR-AUC, ROC-AUC, precision, recall,
F1) plus a separately-labeled final holdout evaluation on the test split.

In [5]:
parameters = mt.build_model_parameters(pipeline, feature_columns)
training_metrics = mt.build_training_metrics(pipeline, X_train, y_train)
validation_metrics = mt.build_validation_metrics(pipeline, X_val, y_val)
holdout_metrics = mt.build_holdout_test_metrics(pipeline, X_test, y_test)
tags = mt.build_reproducibility_tags(feature_columns)

print("Parameters:")
for k, v in parameters.items():
    print(f"  {k}: {v}")

print("\nValidation metrics (used for Phase 7 selection):")
for k, v in validation_metrics.items():
    print(f"  {k}: {v}")

print("\nFinal holdout (test) metrics -- reporting only, NOT used for selection:")
for k, v in holdout_metrics.items():
    print(f"  {k}: {v}")

Parameters:
  model_type: LogisticRegression
  C: 0.01
  class_weight: balanced
  solver: lbfgs
  random_seed: 42
  raw_feature_count: 131
  transformed_feature_count: 142
  train_start: 2025-01-31
  train_end: 2025-11-30
  validation_start: 2025-12-31
  validation_end: 2026-02-28
  training_logged_at: 2026-09-23T06:07:34.577600+00:00

Validation metrics (used for Phase 7 selection):
  val_roc_auc: 0.6233757925318529
  val_pr_auc: 0.02845674366093909
  val_precision: 0.020873574677550005
  val_recall: 0.7477678571428571
  val_f1: 0.04061344486876402
  val_true_negatives: 10175
  val_false_positives: 15714
  val_false_negatives: 113
  val_true_positives: 335
  val_positive_prediction_rate: 0.6093708470972397
  val_threshold: 0.5

Final holdout (test) metrics -- reporting only, NOT used for selection:
  test_roc_auc: 0.671845616476149
  test_pr_auc: 0.038067411922141324
  test_precision: 0.020174286737495523
  test_recall: 0.8407960199004975
  test_f1: 0.039403124271391934
  test_true_ne

## 5. Log Parameters, Metrics, Artifacts, and the Model in One Run

`log_model_run` starts an MLflow run and logs everything in one place:
parameters, metrics (train/validation/holdout, all prefixed so they can
never be confused with each other), reproducibility tags, the small
aggregated report artifacts (`reports/baseline_results.csv`,
`model_comparison_results.csv`, `tuning_results.csv`,
`shap_feature_importance.csv`, `recommendation_summary.csv`), and the
model itself using MLflow's sklearn flavor.

`reports/customer_recommendations.csv` (Phase 9's full-population,
multi-tens-of-MB inference output) is deliberately **not** logged here --
it isn't needed to reproduce or audit the model, and MLflow's tracking
store shouldn't become a second copy of a large generated dataset.

In [6]:
from mlflow.models import infer_signature

input_example = X_val.iloc[:5]
signature = infer_signature(input_example, pipeline.predict_proba(input_example)[:, 1])

run_result = mt.log_model_run(
    pipeline=pipeline,
    run_name="phase7-logistic-regression-frozen",
    parameters=parameters,
    metrics={**training_metrics, **validation_metrics, **holdout_metrics},
    tags=tags,
    signature=signature,
    input_example=input_example,
    config=config,
)

print(f"Run ID: {run_result['run_id']}")
print(f"Model URI: {run_result['model_uri']}")
print(f"Artifact URI: {run_result['artifact_uri']}")
print(f"Logged artifact files: {[p.name for p in run_result['logged_artifact_paths']]}")

/Users/dhananjaymandal/customer-retention-intelligence/.venv/lib/python3.14/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


2026/09/23 11:37:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run ID: 476233cecce9442c84e0b4ff3c1d5d54
Model URI: models:/m-beb0e30f078b44c29e15607c4582514c
Artifact URI: /Users/dhananjaymandal/customer-retention-intelligence/mlartifacts/476233cecce9442c84e0b4ff3c1d5d54/artifacts
Logged artifact files: ['baseline_results.csv', 'model_comparison_results.csv', 'tuning_results.csv', 'shap_feature_importance.csv', 'recommendation_summary.csv']


## 6. Inspect the Logged Run

In [7]:
run = mlflow.get_run(run_result["run_id"])
print(f"Status: {run.info.status}")
print(f"\nParams logged ({len(run.data.params)}):")
for k, v in sorted(run.data.params.items()):
    print(f"  {k} = {v}")
print(f"\nMetrics logged ({len(run.data.metrics)}):")
for k, v in sorted(run.data.metrics.items()):
    print(f"  {k} = {v:.4f}")

Status: FINISHED

Params logged (12):
  C = 0.01
  class_weight = balanced
  model_type = LogisticRegression
  random_seed = 42
  raw_feature_count = 131
  solver = lbfgs
  train_end = 2025-11-30
  train_start = 2025-01-31
  training_logged_at = 2026-09-23T06:07:34.577600+00:00
  transformed_feature_count = 142
  validation_end = 2026-02-28
  validation_start = 2025-12-31

Metrics logged (33):
  test_f1 = 0.0394
  test_false_negatives = 64.0000
  test_false_positives = 16416.0000
  test_positive_prediction_rate = 0.6716
  test_pr_auc = 0.0381
  test_precision = 0.0202
  test_recall = 0.8408
  test_roc_auc = 0.6718
  test_threshold = 0.5000
  test_true_negatives = 8127.0000
  test_true_positives = 338.0000
  train_f1 = 0.0289
  train_false_negatives = 321.0000
  train_false_positives = 33230.0000
  train_positive_prediction_rate = 0.3949
  train_pr_auc = 0.0180
  train_precision = 0.0148
  train_recall = 0.6085
  train_roc_auc = 0.6488
  train_threshold = 0.5000
  train_true_negatives =

## 7. Register the Model

Registers the just-logged model in the MLflow Model Registry under
`configs/mlflow_config.yaml`'s `registry.model_name`
(`customer-churn-logistic-regression`).

In [8]:
model_version = mt.register_model(run_result["model_uri"], config=config)

print(f"Registered model name: {model_version.name}")
print(f"Version: {model_version.version}")
print(f"Source run ID: {model_version.run_id}")
print(f"Status: {model_version.status}")
print(f"Creation timestamp: {model_version.creation_timestamp}")

Registered model name: customer-churn-logistic-regression
Version: 1
Source run ID: 476233cecce9442c84e0b4ff3c1d5d54
Status: READY
Creation timestamp: 1790143659190


Successfully registered model 'customer-churn-logistic-regression'.
Created version '1' of model 'customer-churn-logistic-regression'.


## 8. Load the Registered Model

`load_registered_model` defaults to the latest version, and validates the
loaded pipeline against `configs/mlflow_config.yaml`'s `expected_model`
section (type, `C`, `class_weight`) plus the expected transformed feature
count -- it fails clearly rather than silently returning an incompatible
model.

In [9]:
expected_feature_count = model.coef_.shape[1]
loaded_model = mt.load_registered_model(config=config, expected_feature_count=expected_feature_count)

print(f"Loaded model type: {type(loaded_model.named_steps['model']).__name__}")
print(f"Loaded C: {loaded_model.named_steps['model'].C}")
print(f"Loaded class_weight: {loaded_model.named_steps['model'].class_weight}")
print("Compatibility check passed (no exception raised).")

Loaded model type: LogisticRegression
Loaded C: 0.01
Loaded class_weight: balanced
Compatibility check passed (no exception raised).


## 9. Run a Prediction With the Loaded Model

In [10]:
sample_customers = X_val.iloc[:10]
loaded_predictions = loaded_model.predict_proba(sample_customers)[:, 1]

pd.DataFrame({
    "customer_id": splits["validation"]["customer_id"].iloc[:10].values,
    "predicted_probability": loaded_predictions,
})

,customer_id,predicted_probability
0,C000001,0.450274
1,C000001,0.474991
2,C000001,0.524137
3,C000002,0.338841
4,C000002,0.477572
5,C000002,0.636529
6,C000003,0.595103
7,C000003,0.759907
8,C000003,0.841616
9,C000004,0.535249


## 10. Verify the Loaded Model Matches the Original Phase 7 Model

Predictions from the registered-then-reloaded model must be numerically
identical to the original in-memory Phase 7 pipeline's predictions --
proof that registration/reload introduced no drift whatsoever.

In [11]:
original_predictions = pipeline.predict_proba(sample_customers)[:, 1]

np.testing.assert_allclose(original_predictions, loaded_predictions)
print("Predictions match exactly between the original Phase 7 pipeline and the loaded registered model.")

comparison = pd.DataFrame({
    "customer_id": splits["validation"]["customer_id"].iloc[:10].values,
    "original_probability": original_predictions,
    "loaded_probability": loaded_predictions,
    "difference": original_predictions - loaded_predictions,
})
comparison

Predictions match exactly between the original Phase 7 pipeline and the loaded registered model.


,customer_id,original_probability,loaded_probability,difference
0,C000001,0.450274,0.450274,0.0
1,C000001,0.474991,0.474991,0.0
2,C000001,0.524137,0.524137,0.0
3,C000002,0.338841,0.338841,0.0
4,C000002,0.477572,0.477572,0.0
5,C000002,0.636529,0.636529,0.0
6,C000003,0.595103,0.595103,0.0
7,C000003,0.759907,0.759907,0.0
8,C000003,0.841616,0.841616,0.0
9,C000004,0.535249,0.535249,0.0


## 11. Confirm the Phase 7 Artifact Was Never Modified

In [12]:
from src.models.explainability import TUNED_MODEL_PATH
import hashlib

model_bytes = TUNED_MODEL_PATH.read_bytes()
print(f"models/tuned_logistic_regression.joblib size: {len(model_bytes)} bytes")
print(f"SHA-256: {hashlib.sha256(model_bytes).hexdigest()}")
print("\n(Compare this hash across notebook re-runs -- it must never change.)")

models/tuned_logistic_regression.joblib size: 20282 bytes
SHA-256: 4f30f961c51364247ea6f191cd50c519277a0839725e4830acf07bd6c94c8e81

(Compare this hash across notebook re-runs -- it must never change.)


## 12. Inspecting Runs Locally: the MLflow UI

No cloud server is required. From the project root, with the same
tracking URI configured above:

In [13]:
print(mt.get_mlflow_ui_command(config))
print("\nThen open the printed local URL (typically http://127.0.0.1:5000) in a browser.")

mlflow ui --backend-store-uri sqlite:////Users/dhananjaymandal/customer-retention-intelligence/mlruns.db

Then open the printed local URL (typically http://127.0.0.1:5000) in a browser.


## 13. Summary

In [14]:
print("=== Phase 10 MLflow Tracking Summary ===")
print(f"MLflow version        : {mlflow.__version__}")
print(f"Experiment name       : {experiment.name}")
print(f"Experiment ID         : {experiment.experiment_id}")
print(f"Run ID                : {run_result['run_id']}")
print(f"Registry model name   : {model_version.name}")
print(f"Registered version    : {model_version.version}")
print(f"Parameters logged     : {len(run.data.params)}")
print(f"Metrics logged        : {len(run.data.metrics)}")
print(f"Artifacts logged      : {[p.name for p in run_result['logged_artifact_paths']]} + model")
print(f"Inspect with          : {mt.get_mlflow_ui_command(config)}")
print(f"\nValidation PR-AUC (Phase 7 selection metric): {validation_metrics['val_pr_auc']:.4f}")
print(f"Final holdout (test) PR-AUC (reporting only) : {holdout_metrics['test_pr_auc']:.4f}")

=== Phase 10 MLflow Tracking Summary ===
MLflow version        : 3.16.1
Experiment name       : customer-churn-prediction
Experiment ID         : 1
Run ID                : 476233cecce9442c84e0b4ff3c1d5d54
Registry model name   : customer-churn-logistic-regression
Registered version    : 1
Parameters logged     : 12
Metrics logged        : 33
Artifacts logged      : ['baseline_results.csv', 'model_comparison_results.csv', 'tuning_results.csv', 'shap_feature_importance.csv', 'recommendation_summary.csv'] + model
Inspect with          : mlflow ui --backend-store-uri sqlite:////Users/dhananjaymandal/customer-retention-intelligence/mlruns.db

Validation PR-AUC (Phase 7 selection metric): 0.0285
Final holdout (test) PR-AUC (reporting only) : 0.0381
